# Instagram 데이터 정규화 (1NF → 2NF → 3NF)

`instagram_cleaned.csv` 원본은 수정하지 않고, 정규화 단계별 결과를 별도 CSV로 저장한다.
첨부 이미지에 맞춰 문자열 컬럼은 `str`, 카운트 컬럼은 nullable integer `Int64`, 시간 컬럼은 `datetime64[us, Asia/Seoul]`로 읽는다.

In [ ]:
from pathlib import Path
import pandas as pd

def find_project_root(start: Path) -> Path:
    """현재 위치부터 상위 폴더를 탐색해 프로젝트 루트를 찾는다."""
    for candidate in (start, *start.parents):
        source = candidate / '02_csv' / '02_instagram' / 'instagram_cleaned.csv'
        if source.exists():
            return candidate
    raise FileNotFoundError('instagram_cleaned.csv가 있는 프로젝트 루트를 찾지 못했습니다.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / '02_csv' / '02_instagram'
INPUT_PATH = DATA_DIR / 'instagram_cleaned.csv'

INPUT_PATH

In [ ]:
TEXT_COLUMNS = ['id', 'caption', 'media_type', 'permalink', 'keyword']
COUNT_COLUMNS = ['like_count', 'comments_count']

read_dtypes = {column: 'str' for column in TEXT_COLUMNS}
read_dtypes.update({column: 'Int64' for column in COUNT_COLUMNS})

instagram = pd.read_csv(INPUT_PATH, dtype=read_dtypes)
instagram['timestamp'] = (
    pd.to_datetime(instagram['timestamp'], utc=True)
      .dt.tz_convert('Asia/Seoul')
      .astype('datetime64[us, Asia/Seoul]')
)
instagram = instagram[[
    'id', 'caption', 'media_type', 'permalink', 'timestamp',
    'like_count', 'comments_count', 'keyword'
]]

print(instagram.dtypes)
print(f'행 수: {len(instagram):,}')
instagram.head()

## 1정규화(1NF) 기준

- 각 셀은 하나의 원자값만 가진다. 현재 `keyword`도 행마다 하나의 값이므로 원자적이다.
- 반복 컬럼이나 배열형 컬럼이 없도록 원본의 8개 컬럼을 유지한다.
- 수집 관계의 자연키를 `(id, keyword)`로 보고 완전 중복 및 키 중복을 제거한다.
- 현재 원본은 이미 1NF 조건을 충족하므로 구조는 유지되고, 타입과 유일성만 명시적으로 보장한다.

In [ ]:
nf1 = (
    instagram.drop_duplicates()
             .drop_duplicates(subset=['id', 'keyword'], keep='first')
             .reset_index(drop=True)
)

NF1_PATH = DATA_DIR / 'instagram_normalized_1nf.csv'
nf1.to_csv(NF1_PATH, index=False, encoding='utf-8-sig')
print(f'1NF 저장: {NF1_PATH.name} ({len(nf1):,}행)')

## 2정규화(2NF) 기준

1NF 관계의 개념적 복합키를 `(id, keyword)`로 보면, 게시물 속성인 `caption`, `media_type`, `permalink`, `timestamp`, `like_count`, `comments_count`는 키 전체가 아니라 `id`에만 종속된다. 따라서 부분 함수 종속을 제거한다.

- `posts`: `id → 게시물 속성`
- `post_keywords`: 게시물과 수집 키워드의 연결 `(id, keyword)`

현재 데이터에는 `id` 중복이 없지만, 이후 한 게시물이 여러 키워드와 연결되어도 게시물 속성이 반복 저장되지 않는 구조다.

In [ ]:
post_columns = [
    'id', 'caption', 'media_type', 'permalink', 'timestamp',
    'like_count', 'comments_count'
]
nf2_posts = nf1[post_columns].drop_duplicates(subset=['id']).reset_index(drop=True)
nf2_post_keywords = (
    nf1[['id', 'keyword']]
       .drop_duplicates()
       .sort_values(['id', 'keyword'], kind='stable')
       .reset_index(drop=True)
)

NF2_POSTS_PATH = DATA_DIR / 'instagram_normalized_2nf_posts.csv'
NF2_POST_KEYWORDS_PATH = DATA_DIR / 'instagram_normalized_2nf_post_keywords.csv'
nf2_posts.to_csv(NF2_POSTS_PATH, index=False, encoding='utf-8-sig')
nf2_post_keywords.to_csv(NF2_POST_KEYWORDS_PATH, index=False, encoding='utf-8-sig')

print(f'2NF posts 저장: {NF2_POSTS_PATH.name} ({len(nf2_posts):,}행)')
print(f'2NF post_keywords 저장: {NF2_POST_KEYWORDS_PATH.name} ({len(nf2_post_keywords):,}행)')

## 3정규화(3NF) 기준

2NF 구조에서 반복되는 분류값을 독립 엔터티로 분리해 비키 속성이 다른 비키 속성을 설명하는 경로를 없애고, 분류명 변경 시 한 곳만 수정하도록 한다.

- `media_types(media_type_id, media_type)`: 미디어 유형 마스터
- `keywords(keyword_id, keyword)`: 키워드 마스터
- `posts`: 문자열 `media_type` 대신 `media_type_id` 외래키 사용
- `post_keywords`: 문자열 `keyword` 대신 `keyword_id` 외래키 사용

각 테이블의 모든 비키 속성은 해당 테이블의 키에만 직접 종속된다.

In [ ]:
nf3_media_types = pd.DataFrame({
    'media_type': sorted(nf2_posts['media_type'].dropna().unique())
})
nf3_media_types.insert(0, 'media_type_id', pd.Series(range(1, len(nf3_media_types) + 1), dtype='Int64'))

nf3_keywords = pd.DataFrame({
    'keyword': sorted(nf2_post_keywords['keyword'].dropna().unique())
})
nf3_keywords.insert(0, 'keyword_id', pd.Series(range(1, len(nf3_keywords) + 1), dtype='Int64'))

nf3_posts = (
    nf2_posts.merge(nf3_media_types, on='media_type', how='left', validate='many_to_one')
             .drop(columns='media_type')
             [[
                 'id', 'caption', 'media_type_id', 'permalink', 'timestamp',
                 'like_count', 'comments_count'
             ]]
)
nf3_posts['media_type_id'] = nf3_posts['media_type_id'].astype('Int64')

nf3_post_keywords = (
    nf2_post_keywords.merge(nf3_keywords, on='keyword', how='left', validate='many_to_one')
                     .drop(columns='keyword')
                     [['id', 'keyword_id']]
)
nf3_post_keywords['keyword_id'] = nf3_post_keywords['keyword_id'].astype('Int64')

NF3_POSTS_PATH = DATA_DIR / 'instagram_normalized_3nf_posts.csv'
NF3_POST_KEYWORDS_PATH = DATA_DIR / 'instagram_normalized_3nf_post_keywords.csv'
NF3_KEYWORDS_PATH = DATA_DIR / 'instagram_normalized_3nf_keywords.csv'
NF3_MEDIA_TYPES_PATH = DATA_DIR / 'instagram_normalized_3nf_media_types.csv'

nf3_posts.to_csv(NF3_POSTS_PATH, index=False, encoding='utf-8-sig')
nf3_post_keywords.to_csv(NF3_POST_KEYWORDS_PATH, index=False, encoding='utf-8-sig')
nf3_keywords.to_csv(NF3_KEYWORDS_PATH, index=False, encoding='utf-8-sig')
nf3_media_types.to_csv(NF3_MEDIA_TYPES_PATH, index=False, encoding='utf-8-sig')

print(f'3NF posts 저장: {NF3_POSTS_PATH.name} ({len(nf3_posts):,}행)')
print(f'3NF post_keywords 저장: {NF3_POST_KEYWORDS_PATH.name} ({len(nf3_post_keywords):,}행)')
print(f'3NF keywords 저장: {NF3_KEYWORDS_PATH.name} ({len(nf3_keywords):,}행)')
print(f'3NF media_types 저장: {NF3_MEDIA_TYPES_PATH.name} ({len(nf3_media_types):,}행)')

## 무결성 검증

원본을 변경하지 않았는지와 정규화 과정에서 게시물-키워드 관계가 유실되지 않았는지 확인한다.

In [ ]:
assert nf1[['id', 'keyword']].duplicated().sum() == 0
assert nf2_posts['id'].is_unique
assert nf2_post_keywords[['id', 'keyword']].duplicated().sum() == 0
assert nf3_posts['id'].is_unique
assert nf3_keywords['keyword_id'].is_unique and nf3_keywords['keyword'].is_unique
assert nf3_media_types['media_type_id'].is_unique and nf3_media_types['media_type'].is_unique
assert nf3_post_keywords[['id', 'keyword_id']].duplicated().sum() == 0
assert nf3_posts['media_type_id'].notna().all()
assert nf3_post_keywords['keyword_id'].notna().all()

restored_relations = (
    nf3_post_keywords.merge(nf3_keywords, on='keyword_id', validate='many_to_one')
                     [['id', 'keyword']]
                     .sort_values(['id', 'keyword'], kind='stable')
                     .reset_index(drop=True)
)
expected_relations = (
    nf1[['id', 'keyword']]
       .sort_values(['id', 'keyword'], kind='stable')
       .reset_index(drop=True)
)
pd.testing.assert_frame_equal(restored_relations, expected_relations, check_dtype=False)

restored_nf1 = (
    nf3_posts.merge(nf3_media_types, on='media_type_id', validate='many_to_one')
             .merge(nf3_post_keywords, on='id', validate='one_to_many')
             .merge(nf3_keywords, on='keyword_id', validate='many_to_one')
             [[
                 'id', 'caption', 'media_type', 'permalink', 'timestamp',
                 'like_count', 'comments_count', 'keyword'
             ]]
             .sort_values(['id', 'keyword'], kind='stable')
             .reset_index(drop=True)
)
expected_nf1 = (
    nf1.sort_values(['id', 'keyword'], kind='stable')
       .reset_index(drop=True)
)
pd.testing.assert_frame_equal(restored_nf1, expected_nf1, check_dtype=False)

output_paths = [
    NF1_PATH, NF2_POSTS_PATH, NF2_POST_KEYWORDS_PATH,
    NF3_POSTS_PATH, NF3_POST_KEYWORDS_PATH, NF3_KEYWORDS_PATH, NF3_MEDIA_TYPES_PATH
]
assert all(path.exists() and path.stat().st_size > 0 for path in output_paths)

print('정규화 및 무결성 검증 완료')
pd.DataFrame({
    'file': [path.name for path in output_paths],
    'size_bytes': [path.stat().st_size for path in output_paths],
})